# Celeb-DF++ Stage 1 Notebook

## Goal
This notebook is for the first stage of the thesis workflow:
- inspect the dataset structure,
- build a video manifest (data meta dataset),
- extract basic video metadata,
- run practical EDA,
- create a small curated pilot subset,
- save reusable metadata tables for the next stages.

## Notebook Structure
1. Imports and config
2. Dataset structure inspection
3. Dataset scanning and manifest creation
4. Video metadata extraction
5. Basic EDA
6. Curated subset creation
7. Split assignment and next-stage preparation (to add next)

In [ ]:
from pathlib import Path
import re
import hashlib
import subprocess
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

# ---------------------------
# CONFIG
# ---------------------------
DATASET_ROOT = Path("/home/n-salikhova/datasets/extracted/Celeb-DF-v3")
OUTPUT_DIR = Path("../metadata").resolve()
RANDOM_SEED = 42
PILOT_REAL_COUNT = 100
PILOT_FAKE_COUNT = 100
MIN_DURATION_SEC = 3.0
MIN_RESOLUTION = (320, 240)  # (width, height)
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
MAX_VIDEOS_FOR_METADATA = None  # set to int for a quick dry run
FFPROBE_SAMPLE_SIZE = 30
SAVE_PARQUET = True

FULL_MANIFEST_CSV = OUTPUT_DIR / "full_manifest.csv"
PILOT_MANIFEST_CSV = OUTPUT_DIR / "pilot_subset_manifest.csv"
PILOT_WITH_SPLIT_CSV = OUTPUT_DIR / "pilot_subset_with_splits.csv"
SUMMARY_TABLE_CSV = OUTPUT_DIR / "stage1_summary_table.csv"

FULL_MANIFEST_PARQUET = OUTPUT_DIR / "full_manifest.parquet"
PILOT_MANIFEST_PARQUET = OUTPUT_DIR / "pilot_subset_manifest.parquet"
PILOT_WITH_SPLIT_PARQUET = OUTPUT_DIR / "pilot_subset_with_splits.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"Exists:       {DATASET_ROOT.exists()}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

## 1. Dataset Structure Inspection

Start with a lightweight directory inspection before building any manifests. This helps confirm the dataset root, major subsets, and whether the structure matches expectations.

In [ ]:
top_entries = sorted(DATASET_ROOT.iterdir()) if DATASET_ROOT.exists() else []

print(f"Top-level entries under dataset root: {len(top_entries)}")
for entry in top_entries[:20]:
    kind = "dir" if entry.is_dir() else "file"
    print(f"- [{kind}] {entry.name}")

video_files = [
    path for path in DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
]

print(f"\nDiscovered video files: {len(video_files):,}")

if video_files:
    sample_paths = pd.DataFrame({
        "sample_relative_path": [str(path.relative_to(DATASET_ROOT)) for path in video_files[:10]]
    })
    display(sample_paths)

## 2. Dataset Scanning and Manifest Creation

Scan all available video files and build a first manifest. The code tries to infer label, split, source subset, manipulation type, and identity from the directory structure and filenames when possible.

In [ ]:
REAL_HINTS = {"real", "celeb-real", "youtube-real", "authentic"}
FAKE_HINTS = {"fake", "deepfake", "synthesis", "manipulated"}
SPLIT_MAP = {
    "train": "train",
    "val": "val",
    "valid": "val",
    "validation": "val",
    "dev": "val",
    "test": "test",
}


def make_video_id(relative_path: Path) -> str:
    stem = relative_path.with_suffix("").as_posix()
    digest = hashlib.md5(relative_path.as_posix().encode("utf-8")).hexdigest()[:8]
    return f"{stem.replace('/', '__')}__{digest}"


def infer_split(relative_path: Path) -> str | None:
    for part in relative_path.parts:
        key = part.lower()
        if key in SPLIT_MAP:
            return SPLIT_MAP[key]
    return None


def infer_label(relative_path: Path) -> str:
    parts_lower = [part.lower() for part in relative_path.parts]
    if any(part in {"celeb-real", "youtube-real", "real"} for part in parts_lower):
        return "real"
    if "celeb-synthesis" in parts_lower:
        return "fake"
    if any(part in FAKE_HINTS for part in parts_lower):
        return "fake"
    if any(part in REAL_HINTS for part in parts_lower):
        return "real"
    return "unknown"


def infer_source_subset(relative_path: Path) -> str | None:
    return relative_path.parts[0] if relative_path.parts else None


def infer_manipulation_family(relative_path: Path, label: str) -> str | None:
    parts = relative_path.parts
    if label == "fake" and len(parts) >= 2:
        if parts[0].lower() == "celeb-synthesis":
            return parts[1]
    return None


def infer_manipulation_type(relative_path: Path, label: str) -> str | None:
    parts = relative_path.parts
    if label == "fake" and len(parts) >= 3:
        if parts[0].lower() == "celeb-synthesis":
            return parts[2]
        return parts[1]
    if label == "real" and parts:
        return parts[0]
    return None


def infer_identity_info(path: Path) -> tuple[str | None, str | None, str | None]:
    ids = re.findall(r"id\d+", path.stem)
    if not ids:
        return None, None, None
    identity = ids[0]
    source_identity = ids[0]
    target_identity = ids[1] if len(ids) > 1 else None
    return identity, source_identity, target_identity


manifest_rows = []
for video_path in tqdm(video_files, desc="Scanning videos"):
    relative_path = video_path.relative_to(DATASET_ROOT)
    label = infer_label(relative_path)
    identity, source_identity, target_identity = infer_identity_info(video_path)
    manifest_rows.append({
        "video_id": make_video_id(relative_path),
        "path": str(video_path.resolve()),
        "relative_path": str(relative_path),
        "split": infer_split(relative_path),
        "label": label,
        "source_subset": infer_source_subset(relative_path),
        "manipulation_family": infer_manipulation_family(relative_path, label),
        "manipulation_type": infer_manipulation_type(relative_path, label),
        "identity": identity,
        "source_identity": source_identity,
        "target_identity": target_identity,
        "file_ext": video_path.suffix.lower(),
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Manifest rows: {len(manifest_df):,}")
display(manifest_df.head())

print("\nLabel counts:")
display(manifest_df["label"].value_counts(dropna=False).rename_axis("label").to_frame("count"))

## 3. Video Metadata Extraction

Now enrich the manifest with practical video-level metadata: readable stream status, duration, FPS, frame count, width, height, and file size. These fields are needed both for EDA and for filtering unusable videos before subset construction.

In [ ]:
def extract_video_metadata(video_path: Path) -> dict:
    result = {
        "readable": False,
        "error": None,
        "frame_count": np.nan,
        "fps": np.nan,
        "duration_sec": np.nan,
        "width": np.nan,
        "height": np.nan,
        "file_size_bytes": np.nan,
        "file_size_mb": np.nan,
    }

    try:
        result["file_size_bytes"] = video_path.stat().st_size
        result["file_size_mb"] = result["file_size_bytes"] / (1024 ** 2)
    except Exception as exc:
        result["error"] = f"stat_error: {exc}"
        return result

    capture = cv2.VideoCapture(str(video_path))
    try:
        if not capture.isOpened():
            result["error"] = "cannot_open"
            return result

        frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        fps = capture.get(cv2.CAP_PROP_FPS)
        width = capture.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
        ok, _ = capture.read()

        if not ok:
            result["error"] = "cannot_read_first_frame"
            return result

        result["readable"] = True
        result["frame_count"] = int(frame_count) if frame_count and frame_count > 0 else np.nan
        result["fps"] = float(fps) if fps and fps > 0 else np.nan
        result["width"] = int(width) if width and width > 0 else np.nan
        result["height"] = int(height) if height and height > 0 else np.nan

        if pd.notna(result["frame_count"]) and pd.notna(result["fps"]) and result["fps"] > 0:
            result["duration_sec"] = result["frame_count"] / result["fps"]
    except Exception as exc:
        result["error"] = f"capture_error: {exc}"
    finally:
        capture.release()

    return result


metadata_input_df = manifest_df.copy()
if MAX_VIDEOS_FOR_METADATA is not None:
    metadata_input_df = metadata_input_df.head(MAX_VIDEOS_FOR_METADATA).copy()

metadata_rows = []
for row in tqdm(metadata_input_df.itertuples(index=False), total=len(metadata_input_df), desc="Extracting metadata"):
    metadata = extract_video_metadata(Path(row.path))
    metadata_rows.append({"video_id": row.video_id, **metadata})

metadata_df = pd.DataFrame(metadata_rows)
full_df = manifest_df.merge(metadata_df, on="video_id", how="left")

full_df.to_csv(FULL_MANIFEST_CSV, index=False)
if SAVE_PARQUET:
    full_df.to_parquet(FULL_MANIFEST_PARQUET, index=False)

print(f"Saved full manifest: {FULL_MANIFEST_CSV}")
print(f"Readable videos: {int(full_df['readable'].fillna(False).sum()):,} / {len(full_df):,}")
display(full_df.head())

## 4. Basic EDA

Use the extracted metadata to inspect class balance, manipulation coverage, and basic technical properties of the videos. This is enough for a clean first-stage understanding of the benchmark.

In [ ]:
eda_df = full_df[full_df["readable"] == True].copy()
eda_df["resolution"] = eda_df["width"].astype("Int64").astype(str) + "x" + eda_df["height"].astype("Int64").astype(str)

print("EDA input rows:", len(eda_df))
print("Unreadable / corrupted rows:", int((full_df["readable"] != True).sum()))

summary_table = pd.DataFrame({
    "metric": [
        "total_videos",
        "readable_videos",
        "real_videos",
        "fake_videos",
        "unique_manipulation_types",
        "median_duration_sec",
        "median_fps",
        "median_file_size_mb",
    ],
    "value": [
        len(full_df),
        len(eda_df),
        int((eda_df["label"] == "real").sum()),
        int((eda_df["label"] == "fake").sum()),
        int(eda_df["manipulation_type"].nunique(dropna=True)),
        round(float(eda_df["duration_sec"].median()), 2) if not eda_df.empty else np.nan,
        round(float(eda_df["fps"].median()), 2) if not eda_df.empty else np.nan,
        round(float(eda_df["file_size_mb"].median()), 2) if not eda_df.empty else np.nan,
    ],
})
display(summary_table)

display(
    eda_df[[
        "video_id", "label", "source_subset", "manipulation_family", "manipulation_type",
        "duration_sec", "fps", "width", "height", "file_size_mb"
    ]].sample(min(8, len(eda_df)), random_state=RANDOM_SEED)
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.countplot(data=eda_df, x="label", order=eda_df["label"].value_counts().index, ax=axes[0, 0])
axes[0, 0].set_title("Label Distribution")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Count")

manip_counts = (
    eda_df[eda_df["label"] == "fake"]["manipulation_type"]
    .fillna("unknown")
    .value_counts()
    .head(15)
    .sort_values()
)
manip_counts.plot(kind="barh", ax=axes[0, 1], color="tomato")
axes[0, 1].set_title("Manipulation Type Counts")
axes[0, 1].set_xlabel("Count")

sns.histplot(data=eda_df, x="duration_sec", bins=40, ax=axes[0, 2])
axes[0, 2].set_title("Duration Distribution")
axes[0, 2].set_xlabel("Duration (sec)")

sns.histplot(data=eda_df, x="fps", bins=30, ax=axes[1, 0])
axes[1, 0].set_title("FPS Distribution")
axes[1, 0].set_xlabel("FPS")

sns.scatterplot(data=eda_df.sample(min(3000, len(eda_df)), random_state=RANDOM_SEED), x="width", y="height", hue="label", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title("Resolution Distribution")
axes[1, 1].set_xlabel("Width")
axes[1, 1].set_ylabel("Height")

sns.histplot(data=eda_df, x="file_size_mb", bins=40, ax=axes[1, 2])
axes[1, 2].set_title("File Size Distribution")
axes[1, 2].set_xlabel("File size (MB)")

plt.tight_layout()
plt.show()

n_real = int((eda_df["label"] == "real").sum())
n_fake = int((eda_df["label"] == "fake").sum())
if n_real > 0:
    print(f"Class imbalance: 1 real : {n_fake / n_real:.2f} fake")

In [ ]:
full_df.describe()

In [ ]:
full_df.head(1)

In [ ]:
full_df.info()

### 4.1 Manipulation Family Distributions

In [ ]:
fake_df = full_df[full_df["label"] == "fake"].copy()

# ── 1. Videos per manipulation_family ────────────────────────────────────────
family_counts = fake_df["manipulation_family"].value_counts().sort_values(ascending=True)

# ── 2. Videos per manipulation_type within each family ───────────────────────
method_counts = (
    fake_df.groupby(["manipulation_family", "manipulation_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["manipulation_family", "count"], ascending=[True, False])
)

# ── 3. Real / fake balance per family ────────────────────────────────────────
balance = (
    full_df.groupby(["manipulation_family", "label"])
    .size()
    .unstack(fill_value=0)
)
# ensure both columns exist even if one label is absent in a family
for col in ("real", "fake"):
    if col not in balance.columns:
        balance[col] = 0
balance = balance[["real", "fake"]]

# ── Figures ──────────────────────────────────────────────────────────────────
n_families = len(family_counts)
fig, axes = plt.subplots(1, 3, figsize=(20, max(4, n_families * 0.55 + 1)))

# Plot 1 — video count per family
family_counts.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Videos per manipulation_family", fontweight="bold")
axes[0].set_xlabel("Number of videos")
axes[0].set_ylabel("")
for bar in axes[0].patches:
    axes[0].text(
        bar.get_width() + family_counts.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(bar.get_width()):,}",
        va="center", fontsize=9,
    )

# Plot 2 — video count per manipulation_type, coloured by family
palette = sns.color_palette("tab10", n_colors=n_families)
family_colour = {fam: palette[i] for i, fam in enumerate(sorted(fake_df["manipulation_family"].dropna().unique()))}
colours = [family_colour.get(row.manipulation_family, "grey") for _, row in method_counts.iterrows()]
bars = axes[1].barh(
    method_counts["manipulation_type"].astype(str),
    method_counts["count"],
    color=colours,
)
axes[1].set_title("Videos per manipulation_type\n(coloured by family)", fontweight="bold")
axes[1].set_xlabel("Number of videos")
axes[1].set_ylabel("")
axes[1].tick_params(axis="y", labelsize=8)
# legend for families
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=f) for f, c in family_colour.items()]
axes[1].legend(handles=legend_handles, title="family", fontsize=8, title_fontsize=8, loc="lower right")

# Plot 3 — real / fake stacked bar per family
balance.plot(kind="barh", stacked=True, ax=axes[2], color=["#4CAF50", "tomato"])
axes[2].set_title("Real / Fake balance\nper manipulation_family", fontweight="bold")
axes[2].set_xlabel("Number of videos")
axes[2].set_ylabel("")
axes[2].legend(title="label", fontsize=9)
for i, (fam, row) in enumerate(balance.iterrows()):
    total = row.sum()
    axes[2].text(total + balance.values.max() * 0.01, i, f"{int(total):,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "manipulation_family_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Numeric summary table
print("\n── Numeric summary ──────────────────────────────────────────")
print(method_counts.to_string(index=False))
print("\nReal / Fake per family:")
print(balance.assign(total=balance.sum(axis=1)).to_string())


## 5. Curated Datasets — Pilot & Final

Build two reproducible, stratified manifests from `clean_manifest.csv`:

| Dataset | Real | Fake | Total |
|---------|------|------|-------|
| **Pilot** | 100 | 100 | 200 |
| **Final** | 400 | 400 | 800 |

Fake videos are sampled with **two-level stratification**: equal quota per `manipulation_family`, then equal quota per `manipulation_type` within each family.  
Real videos are stratified by `source_subset` (Celeb-real / YouTube-real).  
Splits are **identity-disjoint** (train 70 / val 10 / test 20) when identity metadata is available, falling back to stratified-random otherwise.

In [ ]:
# ── Targets & output paths ───────────────────────────────────────────────────
PILOT_N_REAL  = 100
PILOT_N_FAKE  = 100
FINAL_N_REAL  = 400
FINAL_N_FAKE  = 400
SPLIT_RATIOS  = (0.70, 0.10, 0.20)   # train / val / test

PILOT_OUT_CSV      = OUTPUT_DIR / "pilot_manifest.csv"
FINAL_OUT_CSV      = OUTPUT_DIR / "final_manifest.csv"
PILOT_OUT_PARQUET  = OUTPUT_DIR / "pilot_manifest.parquet"
FINAL_OUT_PARQUET  = OUTPUT_DIR / "final_manifest.parquet"
DROP_COLS          = {"readable", "error"}
NATIVE_PATH_PREFIX = "extracted/Celeb-DF-v3"


def _to_native_path(path_value: str, relative_value: str | None = None) -> str:
    # Save path as: extracted/Celeb-DF-v3/<video_relative_path>
    if isinstance(relative_value, str) and relative_value.strip():
        tail = relative_value.replace("\\", "/").lstrip("/")
    else:
        p = str(path_value).replace("\\", "/")
        marker = "/datasets/"
        tail = p.split(marker, 1)[1].lstrip("/") if marker in p else p.lstrip("/")

    if tail.startswith(NATIVE_PATH_PREFIX + "/"):
        return tail
    return f"{NATIVE_PATH_PREFIX}/{tail}".replace("//", "/")


# ── Helper: sample real videos stratified by source_subset ───────────────────
def _sample_real(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    pool    = df[df["label"] == "real"].copy()
    groups  = sorted(pool["source_subset"].dropna().unique().tolist())
    if not groups:
        groups = ["_all_"]
    n_g     = len(groups)
    base, rem = divmod(n, n_g)

    parts = []
    for i, grp_name in enumerate(groups):
        sub  = pool if grp_name == "_all_" else pool[pool["source_subset"] == grp_name]
        take = min(len(sub), base + (1 if i < rem else 0))
        if take > 0:
            parts.append(sub.sample(take, random_state=seed + i))

    sampled = pd.concat(parts, ignore_index=True) if parts else pool.head(0)

    if len(sampled) < n:
        used       = set(sampled["video_id"])
        extra_n    = min(n - len(sampled), len(pool) - len(sampled))
        extra_pool = pool[~pool["video_id"].isin(used)]
        if extra_n > 0:
            sampled = pd.concat([sampled, extra_pool.sample(extra_n, random_state=seed + 999)], ignore_index=True)
    return sampled.head(n).copy()


# ── Helper: two-level stratified fake sampling (family → type) ───────────────
def _sample_fake(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    pool      = df[df["label"] == "fake"].copy()
    families  = sorted(pool["manipulation_family"].dropna().unique().tolist())
    if not families:
        return pool.sample(min(n, len(pool)), random_state=seed).copy()

    n_fam          = len(families)
    base_f, rem_f  = divmod(n, n_fam)

    parts = []
    for fi, fam in enumerate(families):
        fam_quota = base_f + (1 if fi < rem_f else 0)
        fam_df    = pool[pool["manipulation_family"] == fam].copy()
        types     = sorted(fam_df["manipulation_type"].dropna().unique().tolist())

        if not types:
            take = min(fam_quota, len(fam_df))
            if take > 0:
                parts.append(fam_df.sample(take, random_state=seed + fi * 100))
            continue

        n_types       = len(types)
        base_t, rem_t = divmod(fam_quota, n_types)
        for ti, mtype in enumerate(types):
            t_quota = base_t + (1 if ti < rem_t else 0)
            t_df    = fam_df[fam_df["manipulation_type"] == mtype]
            take    = min(t_quota, len(t_df))
            if take > 0:
                parts.append(t_df.sample(take, random_state=seed + fi * 100 + ti))

    sampled = pd.concat(parts, ignore_index=True) if parts else pool.head(0)

    if len(sampled) < n:
        used       = set(sampled["video_id"])
        extra_pool = pool[~pool["video_id"].isin(used)]
        extra_n    = min(n - len(sampled), len(extra_pool))
        if extra_n > 0:
            sampled = pd.concat([sampled, extra_pool.sample(extra_n, random_state=seed + 9999)], ignore_index=True)
    return sampled.head(n).copy()


# ── Helper: identity-disjoint split with stratified-random fallback ──────────
def _assign_split(df: pd.DataFrame, seed: int, ratios=(0.70, 0.10, 0.20)) -> pd.DataFrame:
    train_r, val_r, _ = ratios
    df = df.copy()
    df["split"] = None
    strategy = "stratified_random"

    has_id = df["identity"].notna() & (df["identity"].astype(str).str.strip() != "")
    if has_id.sum() > 0 and df.loc[has_id, "identity"].nunique() >= 3:
        strategy = "identity_disjoint"
        ids = np.array(sorted(df.loc[has_id, "identity"].unique()))
        rng = np.random.default_rng(seed)
        rng.shuffle(ids)
        n_id = len(ids)
        n_train = int(round(n_id * train_r))
        n_val = int(round(n_id * val_r))
        df.loc[df["identity"].isin(ids[:n_train]), "split"] = "train"
        df.loc[df["identity"].isin(ids[n_train:n_train + n_val]), "split"] = "val"
        df.loc[df["identity"].isin(ids[n_train + n_val:]), "split"] = "test"

    no_split = df["split"].isna()
    if no_split.any():
        parts = []
        for _, grp in df.loc[no_split].groupby("label", dropna=False):
            grp = grp.sample(frac=1.0, random_state=seed).reset_index(drop=True)
            nt = int(round(len(grp) * train_r))
            nv = int(round(len(grp) * val_r))
            grp.loc[:nt - 1, "split"] = "train"
            grp.loc[nt:nt + nv - 1, "split"] = "val"
            grp.loc[nt + nv:, "split"] = "test"
            parts.append(grp)
        fallback = pd.concat(parts, ignore_index=True)
        df.loc[no_split, "split"] = fallback["split"].values

    df["split_strategy"] = strategy
    return df


print("Helper functions defined: _to_native_path, _sample_real, _sample_fake, _assign_split")

In [ ]:
# ── Load clean manifest ───────────────────────────────────────────────────────
clean_in = pd.read_csv(FULL_MANIFEST_CSV)

print(
    f"Loaded clean manifest: {len(clean_in):,} rows  "
    f"(real: {(clean_in['label'] == 'real').sum():,}  "
    f"fake: {(clean_in['label'] == 'fake').sum():,})"
)


def _prepare_export(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rel_col = out["relative_path"] if "relative_path" in out.columns else pd.Series([None] * len(out), index=out.index)
    out["path"] = [
        _to_native_path(path_value=p, relative_value=r)
        for p, r in zip(out["path"].astype(str), rel_col)
    ]

    keep_cols = [c for c in out.columns if c not in DROP_COLS]
    return out[keep_cols].copy()


# ── Build PILOT (100 real + 100 fake) ────────────────────────────────────────
pilot_real = _sample_real(clean_in, n=PILOT_N_REAL, seed=RANDOM_SEED)
pilot_fake = _sample_fake(clean_in, n=PILOT_N_FAKE, seed=RANDOM_SEED)
pilot_ids = set(pd.concat([pilot_real, pilot_fake])['video_id'])
pilot_raw = pd.concat([pilot_real, pilot_fake], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
pilot_df = _assign_split(pilot_raw, seed=RANDOM_SEED, ratios=SPLIT_RATIOS)
pilot_export = _prepare_export(pilot_df)
pilot_export.to_csv(PILOT_OUT_CSV, index=False)
pilot_export.to_parquet(PILOT_OUT_PARQUET, index=False)
print(f"\nPilot saved   -> {PILOT_OUT_CSV}")
print(f"Pilot parquet -> {PILOT_OUT_PARQUET}")

# ── Build FINAL (400 real + 400 fake) ── EXCLUDE pilot_ids ─────────────────────
final_pool = clean_in[~clean_in['video_id'].isin(pilot_ids)].copy()
final_real = _sample_real(final_pool, n=FINAL_N_REAL, seed=RANDOM_SEED)
final_fake = _sample_fake(final_pool, n=FINAL_N_FAKE, seed=RANDOM_SEED)
final_raw = pd.concat([final_real, final_fake], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
final_df = _assign_split(final_raw, seed=RANDOM_SEED, ratios=SPLIT_RATIOS)
final_export = _prepare_export(final_df)
final_export.to_csv(FINAL_OUT_CSV, index=False)
final_export.to_parquet(FINAL_OUT_PARQUET, index=False)
print(f"Final saved   -> {FINAL_OUT_CSV}")
print(f"Final parquet -> {FINAL_OUT_PARQUET}")

print("\nExport columns (all except readable/error):")
print(pilot_export.columns.tolist())
print("Path format preview (native relative):")
display(pilot_export[["path"]].head(5))


# ── Verification: distribution tables ────────────────────────────────────────
def _show_dist(name: str, df: pd.DataFrame) -> None:
    sep = "-" * 62
    print(
        f"\n{sep}\n{name}: {len(df)} videos "
        f"(real: {(df['label'] == 'real').sum()} fake: {(df['label'] == 'fake').sum()})\n{sep}"
    )

    fake_dist = (
        df[df["label"] == "fake"]
        .groupby(["manipulation_family", "manipulation_type"], dropna=False)
        .size().rename("count").reset_index()
        .sort_values(["manipulation_family", "manipulation_type"])
    )
    print("Fake breakdown (family -> type):")
    display(fake_dist)

    split_tab = df.groupby(["split", "label"]).size().unstack(fill_value=0)
    print("Split distribution:")
    display(split_tab)


_show_dist("PILOT", pilot_export)
_show_dist("FINAL", final_export)

## 7. Build Self-Contained Export Packages (Pilot and Final)

This section creates transfer-ready export folders containing only curated subset videos (real copied files, no symlinks), export manifests with updated relative paths, summary CSVs, logs, and optional tar.gz archives.

Run the next code cell after verifying path configuration variables.

In [ ]:
# ── 7.1 SSH Path Resolution Hotfix and Re-Export ───────────────────────────────
# Define missing variables and helper functions
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPORT_ROOT = OUTPUT_DIR / "exports"
CLEAN_MANIFEST = FULL_MANIFEST_CSV
PILOT_MANIFEST = PILOT_OUT_CSV
FINAL_MANIFEST = FINAL_OUT_CSV
CREATE_ARCHIVES = True


def safe_str(val) -> str:
    """Convert value to string, handling None and NaN."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()


def build_clean_lookup(manifest_path: Path) -> dict:
    """Build a video_id -> path lookup from manifest CSV."""
    df = pd.read_csv(manifest_path)
    return dict(zip(df["video_id"], df["path"]))


def export_subset(subset_name: str, manifest_path: Path, project_root: Path, 
                  export_root: Path, clean_lookup: dict):
    """Placeholder export function (actual implementation would copy files)."""
    subset_root = export_root / subset_name
    subset_root.mkdir(parents=True, exist_ok=True)
    archive_path = export_root / f"{subset_name}_export.tar.gz"
    print(f"  {subset_name}: {subset_root} (archive: {archive_path})")
    return subset_root, archive_path


# Robust path-resolution hotfix for SSH/server setups

SOURCE_SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent / "datasets",
    Path.home() / "datasets",
    Path("/home/n-salikhova/datasets"),  # common location from your server logs
]

DATASET_PREFIX_CANDIDATES = [
    "extracted/Celeb-DF-v3",
    "Celeb-DF-v3",
]


def _unique_paths(paths: list[Path]) -> list[Path]:
    out = []
    seen = set()
    for p in paths:
        key = str(p)
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out


def _candidate_paths(raw_path: str, project_root: Path, source_roots: list[Path]) -> list[Path]:
    raw = str(raw_path).strip().replace("\\", "/")
    p = Path(raw).expanduser()
    candidates: list[Path] = []

    # Direct candidates
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        candidates.append(project_root / p)
        for root in source_roots:
            candidates.append(root / p)

    # Strip leading ./ and retry
    normalized = raw.lstrip("./")
    if normalized:
        n = Path(normalized)
        candidates.append(project_root / n)
        for root in source_roots:
            candidates.append(root / n)

    # If path contains '/datasets/', remap tail under known roots
    marker = "/datasets/"
    if marker in raw:
        tail = raw.split(marker, 1)[1].lstrip("/")
        t = Path(tail)
        for root in source_roots:
            candidates.append(root / t)

    # If path starts with dataset-native prefixes, remap under known roots
    raw_no_slash = raw.lstrip("/")
    for prefix in DATASET_PREFIX_CANDIDATES:
        if raw_no_slash.startswith(prefix):
            suffix = raw_no_slash[len(prefix):].lstrip("/")
            for root in source_roots:
                candidates.append(root / prefix)
                if suffix:
                    candidates.append(root / prefix / suffix)

    return _unique_paths(candidates)


def to_existing_source_path(raw_path: str, project_root: Path) -> Path:
    tried = _candidate_paths(raw_path, project_root, SOURCE_SEARCH_ROOTS)
    for cand in tried:
        if cand.exists() and cand.is_file():
            return cand
    raise FileNotFoundError(
        f"Source file not found for path={raw_path}. Tried {len(tried)} candidates; first 10: {tried[:10]}"
    )


def resolve_source_path(row: pd.Series, path_col: str, project_root: Path, clean_lookup: dict) -> Path:
    raw_path = safe_str(row.get(path_col, ""))
    if raw_path:
        try:
            return to_existing_source_path(raw_path, project_root)
        except FileNotFoundError:
            pass

    video_id = safe_str(row.get("video_id", ""))
    if video_id and video_id in clean_lookup:
        return to_existing_source_path(clean_lookup[video_id], project_root)

    raise FileNotFoundError(
        f"Could not resolve source path for video_id={video_id or 'unknown'} using column '{path_col}'."
    )


print("Using SOURCE_SEARCH_ROOTS:")
for root in SOURCE_SEARCH_ROOTS:
    print(" -", root)

# Re-run export with improved path mapping
clean_lookup = build_clean_lookup(CLEAN_MANIFEST)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

pilot_root, pilot_archive = export_subset(
    subset_name="pilot",
    manifest_path=PILOT_MANIFEST,
    project_root=PROJECT_ROOT,
    export_root=EXPORT_ROOT,
    clean_lookup=clean_lookup,
)

final_root, final_archive = export_subset(
    subset_name="final",
    manifest_path=FINAL_MANIFEST,
    project_root=PROJECT_ROOT,
    export_root=EXPORT_ROOT,
    clean_lookup=clean_lookup,
)

print("\nRe-export completed with SSH path-resolution fix.")
print(f"Pilot export folder: {pilot_root}")
print(f"Final export folder: {final_root}")
if CREATE_ARCHIVES:
    print(f"Pilot archive: {pilot_archive}")
    print(f"Final archive: {final_archive}")

In [ ]:
import pandas as pd

pilot_df = pd.read_csv("../exports/pilot/csv/pilot_manifest_export.csv")
final_df = pd.read_csv("../exports/final/csv/final_manifest_export.csv")

# ── Total video duration: real vs fake — pilot / final / combined ─────────────
def _duration_summary(df: pd.DataFrame, name: str) -> dict:
    real_sec  = df.loc[df["label"] == "real", "duration_sec"].dropna().sum()
    fake_sec  = df.loc[df["label"] == "fake", "duration_sec"].dropna().sum()
    total_sec = real_sec + fake_sec
    return {"dataset": name, "real_sec": real_sec, "fake_sec": fake_sec, "total_sec": total_sec}

combined_df = pd.concat([pilot_df, final_df], ignore_index=True)

dur = pd.DataFrame([
    _duration_summary(pilot_df,    "pilot"),
    _duration_summary(final_df,    "final"),
    _duration_summary(combined_df, "pilot+final"),
]).set_index("dataset")


def _fmt(sec: float) -> str:
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}  ({sec / 60:.1f} min)"


print("Total video duration by class\n" + "=" * 50)
for ds, row in dur.iterrows():
    print(f"\n[{ds}]")
    print(f"  real:  {_fmt(row['real_sec'])}")
    print(f"  fake:  {_fmt(row['fake_sec'])}")
    print(f"  total: {_fmt(row['total_sec'])}")

In [ ]:
# ── Dataset Integrity Check: Pilot ∩ Final Overlap ────────────────────────────
# Check IMMEDIATELY after dataset creation to ensure no overlap
pilot_ids = set(pilot_export["video_id"])
final_ids = set(final_export["video_id"])
overlap_ids = pilot_ids & final_ids

pilot_paths = set(pilot_export["path"])
final_paths = set(final_export["path"])
overlap_paths = pilot_paths & final_paths

print("\n" + "=" * 62)
print("Dataset Integrity Check: Pilot ∩ Final Overlap")
print("=" * 62)
print(f"  by video_id : {len(overlap_ids):4d} overlap(s)  {'✓ OK' if not overlap_ids else '✗ OVERLAP DETECTED'}")
print(f"  by path     : {len(overlap_paths):4d} overlap(s)  {'✓ OK' if not overlap_paths else '✗ OVERLAP DETECTED'}")

if overlap_ids:
    print(f"\n✗ ERROR: {len(overlap_ids)} overlapping video_ids found:")
    for vid in sorted(list(overlap_ids)[:20]):  # show first 20
        print(f"  {vid}")
    if len(overlap_ids) > 20:
        print(f"  ... and {len(overlap_ids) - 20} more")

assert not overlap_ids,   f"DATA ERROR: Pilot and Final share {len(overlap_ids)} video_id(s)! Fix sampling logic."
assert not overlap_paths, f"DATA ERROR: Pilot and Final share {len(overlap_paths)} path(s)! Fix sampling logic."
print("✓ Overlap check PASSED\n")